오늘 해야 할 것들
1. 원본 데이터 소개
2. 데이터 전처리
3. 데이터 분석

4. 서울기 거주/비거주 고객의 소비 분석

4-1. 서울기 거주/비거주 고객 수 구하기

4-2. 총 소비액 구하기

4-3. 셩별 소비액 구하기

5. 편의점 소비 정보 분석

5-1. 편의점 소비액 구하기

5-2. 강남구 편의점 소비액 분석하기

5-3. 거주지 소재 편의점 소비액 구하기

###
사용할 함수들

sum() : 결측치를 자동으로 건너띄고 정상적인 값들만 전부 합해준다.
        소비액의 총합을 구할 때 사용할 예정

DataFrame() : 읽어온 데이터를 데이터프레임으로 변환해준다

info() : 데이터프레임의 간략한 정보를 출력

head() : 가장 윗 행의 몇 개만 출력

len() : 길이를 구해준다. 고객의 수를 세는데 사용할 예정

In [1]:
import pandas as pd
import numpy as np

1. 원본 데이터 소개 & 파일 읽어오기

원본 파일은 txt 파일이고, 약 12MB 정도로 크기가 크다.

txt 파일을 메모장으로 열어보면, \t로 구분되는 것을 볼 수 있다.

그러나 아무 조건없이 메모장을 읽으면, 메모장의 가장 윗 줄만 읽게 되어서 비정상적인 값이 나올 뿐 아니라 데이터프레임으로 변환할 수 없다.

따라서 \t를 나누는 기준으로 하고, utf-8으로 인코딩한다.

In [2]:
# df에 인코딩한 파일 저장

file_path = "/content/bc_card_out2020_03.txt"
try:
  with open( file_path, 'r', encoding = 'utf-8') as f:
    df = pd.read_csv(
        f, sep = '\t', low_memory=False
    )
    print('데이터 로드 성공!')
    print(df.head())
except Exception as e:
    print(f'오류 발생: {e}')

데이터 로드 성공!
   REG_YYMM  MEGA_CTY_NO MEGA_CTY_NM  CTY_RGN_NO CTY_RGN_NM  ADMI_CTY_NO  \
0    202003           11       서울특별시        1168        강남구     11680750   
1    202003           11       서울특별시        1129        성북구     11290660   
2    202003           11       서울특별시        1144        마포구     11440555   
3    202003           11       서울특별시        1126        중랑구     11260590   
4    202003           11       서울특별시        1168        강남구     11680630   

  ADMI_CTY_NM  MAIN_BUZ_CODE MAIN_BUZ_DESC  TP_GRP_NO  ... CSTMR_GUBUN  \
0         수서동             30            생활         40  ...         내국인   
1        길음1동             30            생활         70  ...         내국인   
2         아현동             30            생활         40  ...         내국인   
3        상봉2동             30            생활         40  ...         내국인   
4        대치4동             30            생활         62  ...         내국인   

   CSTMR_MEGA_CTY_NO CSTMR_MEGA_CTY_NM CSTMR_CTY_RGN_NO  CSTMR_CTY_RGN_NM  \
0         

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1157597 entries, 0 to 1157596
Data columns (total 23 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   REG_YYMM           1157597 non-null  int64  
 1   MEGA_CTY_NO        1157597 non-null  int64  
 2   MEGA_CTY_NM        1157597 non-null  object 
 3   CTY_RGN_NO         1157597 non-null  int64  
 4   CTY_RGN_NM         1157597 non-null  object 
 5   ADMI_CTY_NO        1157597 non-null  int64  
 6   ADMI_CTY_NM        1157597 non-null  object 
 7   MAIN_BUZ_CODE      1157597 non-null  int64  
 8   MAIN_BUZ_DESC      1157597 non-null  object 
 9   TP_GRP_NO          1157597 non-null  int64  
 10  TP_GRP_NM          1157597 non-null  object 
 11  TP_BUZ_NO          1157597 non-null  int64  
 12  TP_BUZ_NM          1157597 non-null  object 
 13  CSTMR_GUBUN        1157597 non-null  object 
 14  CSTMR_MEGA_CTY_NO  1157597 non-null  int64  
 15  CSTMR_MEGA_CTY_NM  1157597 non-n

In [7]:
len(df)

1157597

df에는 총 23개의 column이 있고, 고객 수는 총 1157597명 이다.

2. 데이터 전처리 단계

In [8]:
df_seoul = df[df['CSTMR_MEGA_CTY_NM'] == '서울특별시']
# 서울 거주 고객들의 정보.
# df 데이터들 중 CSTMR_MEGA_CITY_NM이이 서울특별시인 모든 고객들이 이에 해당한다.

df_not_seoul = df[df['CSTMR_MEGA_CTY_NM'] != '서울특별시']
# 비서울 거주 고객들의 정보.
# df 데이터들 중 CSTMR_MEGA_CITY_NM이 서울특별시가 아닌 모든 고객들이 이에 해당한다.

df_man = df[df['SEX_CTGO_CD'] == 1]
# 여성 고객들의 정보.
# df 데이터들 중 고객의 성별인 SEX_CTGO_CD가 1인 모든 고객이 이에 해당한다.
# SEX_CTGO_CD에서 어느 것이 남성인지는 알 수 없어 편의상 1을 남성으로 지정했다.

df_woman = df[df['SEX_CTGO_CD'] == 2]
# 여성 고객들의 정보.
# df 데이터들 중 고객의 성별인 SEX_CTGO_CD가 2인 모든 고객이 이에 해당한다.
# SEX_CTGO_CD에서 어느 것이 여성인지는 알 수 없어 편의상 2를 여성으로 지정했다.

df_convenience = df[df['TP_BUZ_NM'] == '편 의 점']
# 편의점을 이용한 고객 정보.
# df 데이터들 중 업종 소분류명인 TP_BUZ_NO가 '편 의 점'인 모든 업종이 이에 해당한다.
# 오타가 아니라, 실제 column명이 '편 의 점'으로 되어 있다.

df_convenience_gangnam = df_convenience[df_convenience['CTY_RGN_NM'] == '강남구']
# 강남구 편의점 고객 정보.
# df 데이터들 중 가맹점 소재지의 시군구 이름인 CTY_RGN_NM이 강남구인 모든 고객들이 이에 해당한다.

df_convenience_gangnam_seoul = df_convenience_gangnam[df_convenience_gangnam['CSTMR_MEGA_CTY_NM'] == '서울특별시']
# 강남구 편의점 이용고객 중 서울시 거주자들의 정보.
# df 데이터들 중 CSTMR_MEGA_CITY_NM이 서울특별시인 모든 고객들이 이에 해당한다.

df_convenience_gangnam_not_seoul = df_convenience_gangnam[df_convenience_gangnam['CSTMR_MEGA_CTY_NM'] != '서울특별시']
# 강남 편의점 이용고객 중 서울시 비거주자들의 정보.
# df 데이터들 중 CSTMR_MEGA_CITY_NM이 서울특별시가 아닌 모든 고객들이 이에 해당한다.

df_convenience_local = df_convenience[df_convenience['CTY_RGN_NM'] == df_convenience['CSTMR_CTY_RGN_NM']]
# 거주지 소재 편의점 이용 고객들의 정보.
# df 데이터들 중 편의점의 위치인 CTY_RGN_NM과 CSTMR_CTY_RGN_NM이 같은 데이터들만 추출했다.


3. 데이터 분석

3-1. 서울시 거주/비거주 고객의 소비 분석

len으로는 거주자 혹은 고객의 수를 구한다.

sum()으로는 'AMT' column의 모든 수를 합해 총 소비액을 구한다.

In [23]:
# 서울/비서울 거주자 수 구하기
# len으로 거주자의 수를 구한다
print(f"서울 거주자 수: {len(df_seoul)}")
print(f"비서울 거주자 수: {len(df_not_seoul)}")

서울 거주자 수: 656808
비서울 거주자 수: 500789


In [21]:
# 총 소비액 구하기
# df에서 AMT 열의 모든 수를 합한다
# sum이 결측치를 제외한 값을 산출한다
total = df['AMT'].sum()
print(f"총 소비액: {total}")

총 소비액: 2479327398571.0


In [22]:
# 남자 소비액 구하기
# df_man에서 AMT 열의 모든 수를 합한다
# sum이 결측치를 제외한 값을 산출한다
df_man_total = df_man['AMT'].sum()
print(f"남자 소비액: {df_man_total}")

남자 소비액: 1264541217550.0


In [19]:
# 여자 소비액 구하기
# df_woman에서 AMT 열의 모든 수를 합한다
# sum이 결측치를 제외한 값을 산출한다
df_woman_total = df_woman['AMT'].sum()
print(f"여자 소비액: {df_woman_total}")

여자 소비액: 1214786181021.0


3-2. 편의점 소비 정보 분석

sum()으로 'AMT' column의 모든 수를 합해 총 소비액을 구한다.

In [15]:
# 편의점 총 소비액
# df_convenience에서 AMT 열의 모든 수를 합한다
# sum이 결측치를 제외한 값을 산출한다
total_convenience = df_convenience['AMT'].sum()
print(f"편의점 총 소비액: {total_convenience}")

편의점 총 소비액: 59476156250.0


In [16]:
# 강남 편의점 이용고객 중 서울 거주 고객과 그 소비액
# df_convenience_gangnam_seoul에서 AMT 열의 모든 수를 합한다
# sum이 결측치를 제외한 값을 산출한다
total_convenience_gangnam_seoul = df_convenience_gangnam_seoul['AMT'].sum()
print(f"강남 편의점 이용고객 중 서울 거주 고객의 총 소비액: {total_convenience_gangnam_seoul}")

강남 편의점 이용고객 중 서울 거주 고객의 총 소비액: 4489491554.0


In [17]:
# 강남 편의점 이용고객 중 서울기 비거주 고객과 그 편의점 소비액
# df_convenience_gangnam_not_seoul에서 AMT 열의 모든 수를 합한다
total_convenience_gangnam_not_seoul = df_convenience_gangnam_not_seoul['AMT'].sum()
print(f"강남 편의점 이용고객 중 서울시 비거주 고객의 총 소비액: {total_convenience_gangnam_not_seoul}")

강남 편의점 이용고객 중 서울시 비거주 고객의 총 소비액: 1562255827.0


In [18]:
# 거주지 소재 편의점 이용 고객과 그 소비액 구하기
# df_convenience_local에서 AMT 열의 모든 수를 합한다
total_convenience_local = df_convenience_local['AMT'].sum()
print(f"거주지 소재 편의점 이용고객의 총 소비액: {total_convenience_local}")

거주지 소재 편의점 이용고객의 총 소비액: 32989917735.0
